# **Install Libraries**

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 946.3 kB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.33.1
    Uninstalling requests-2.33.1:
      Successfully uninstalled requests-2.33.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# **Step 1a Indexing (Document Ingestion)**

In [ ]:
video_id = "dxsawQOQ9lU" # only the ID, not full URL
try:
    api = YouTubeTranscriptApi()
    # If you don’t care which language, this returns the “best” one
    transcript_list = api.list(video_id)

    # Flatten it to plain text
    fetched_obj = transcript_list.find_transcript(['en']).fetch()
    transcript_dict = fetched_obj.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_dict)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Hey guys and welcome to Cool Vision. In this video, we're going to be talking about New York City. Some call it the least American city. No wonder 55% of households don't have a car here. It's the most popular city in the US with a population of 8.8 million and 20 million people living in the metro area. Let's learn more about New York. New York City. New York City is situated on a large natural harbor composed of five burrows. The Bronx, Brooklyn, Queens, Staten Island, and of course, Manhattan. Two blocks down and take a right. A block has a literal meaning here. All streets are perpendicular and form a grid. Only the southern tip of the island is knocked out of the clear geometry where the streets don't form a grid and Broadway which runs diagonally across the island. The commissioner's plan of 1811 was the original design for the streets of Manhattan which put in place the rectangular grid plan. Pretty convenient. The basic thing to remember is that avenues run north and south whil

In [ ]:
transcript

'Hey guys and welcome to Cool Vision. In this video, we\'re going to be talking about New York City. Some call it the least American city. No wonder 55% of households don\'t have a car here. It\'s the most popular city in the US with a population of 8.8 million and 20 million people living in the metro area. Let\'s learn more about New York. New York City. New York City is situated on a large natural harbor composed of five burrows. The Bronx, Brooklyn, Queens, Staten Island, and of course, Manhattan. Two blocks down and take a right. A block has a literal meaning here. All streets are perpendicular and form a grid. Only the southern tip of the island is knocked out of the clear geometry where the streets don\'t form a grid and Broadway which runs diagonally across the island. The commissioner\'s plan of 1811 was the original design for the streets of Manhattan which put in place the rectangular grid plan. Pretty convenient. The basic thing to remember is that avenues run north and sou

# **Step 1b**

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
print(type(transcript))
chunks = splitter.create_documents([transcript])

<class 'str'>


In [ ]:
len(chunks)

47

In [ ]:
chunks[46]

Document(metadata={}, page_content='creative people who want to succeed in finance, fashion, and other industries. They say, "If you can make it here, you can make it anywhere." And it\'s true. The sky is the limit. New York has more billionaires than any other city in the world. There were 99 billionaires living here in 2021. City has been shaped by different waves of immigrants, but even today, it is home to more than 3.2 2 million residents born outside the United States, the largest foreignb born population of any city in the world. Let me know what you think about New York City, and I\'ll see you in my next video. Heat. Heat. [Applause] [Music] [Applause] [Music] Heat.')

# **Step 1c & 1d**

In [ ]:
from langchain_core import embeddings
embedding = HuggingFaceEmbeddings(model="thenlper/gte-small")
vector_store = FAISS.from_documents(chunks, embedding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store.index_to_docstore_id

{0: '8b1ed178-9239-4a68-b92f-7d8140aed45f',
 1: '34634c99-cdf0-420c-81e1-935f07e73314',
 2: 'a36c660f-d5a2-4ed4-b769-c2ab8aff229c',
 3: 'adcfecf5-2867-4330-b41c-a211675b5034',
 4: '7b0d581d-a101-43db-80a5-713b13a806d6',
 5: 'ac7f9323-2d12-4b22-988e-5d58005904d8',
 6: '023a811c-b3aa-4871-a1a1-3b744b8d6948',
 7: '055b28e4-714c-4b53-b864-cfde33d25d4f',
 8: '880c6387-bfdc-4805-9ec1-417e6d64ab65',
 9: 'dd2c4a22-d999-471f-9499-cf820008f912',
 10: 'f505209c-269a-4a77-8061-1df4174640e3',
 11: 'c984b20d-da07-4944-866d-231cef57be22',
 12: '677ee42c-1a5e-4797-8cb1-bfdbdb70df36',
 13: 'fbe0f0f6-0718-4db1-88b8-998c94b18151',
 14: 'ac2e9f94-b67d-4a6e-a4ef-8d74a7d46e2a',
 15: '11a24d08-4e74-4264-93a8-101d158514b8',
 16: 'a525f1e3-aa25-4be1-8a4b-3c4c17b22aa7',
 17: '37e247e2-74a9-4b89-9db8-58dd17cd33a2',
 18: '12faced3-2278-455e-862b-c640a5f5b4ff',
 19: '0c2e0297-d9c2-4b7c-ae8b-63edf36b635c',
 20: 'c34ab1c2-8932-4c6d-b122-17f46e81b68b',
 21: '0ae1cffb-cb5d-448f-bf02-5394a3152bd7',
 22: 'b9fe354c-394b-

In [ ]:
vector_store.get_by_ids(['e1f6c992-cc7e-496c-9ac6-0c99247e5d82'])

[Document(id='e1f6c992-cc7e-496c-9ac6-0c99247e5d82', metadata={}, page_content="of the park, resembling the interior of the old refinery. New York City is a city of hundreds of different languages and a variety of communities. Yes, it's a melting pot, but only to a certain extent. Most of its neighborhoods are dominated by a certain ethnicity. You know, it makes sense because people love to be around other people that share a similar cultural background. Let's visit a Jewish area in Brooklyn called Burrow Park. First thing you notice is the abundance of young mothers with strollers. The fertility rate for Orthodox Jews is one of the highest in the world. An average of 6.5 children per woman. For comparison, in Europe is 1.6. Let's talk to some locals. Uh what part of town are we in? You're in the Burough Park section of Brooklyn. And what's it famous for? Jewish neighborhood. What is it that you like mostly about living here? Well, it's the community. you know, it's we're tightknit, we

# Step 2 - **Retrieval**

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b400bbdb140>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke("What is avenue")

[Document(id='dd2c4a22-d999-471f-9499-cf820008f912', metadata={}, page_content="City. Well, the whole world lives here. So, that's number one. Uh when you can't travel, you're in New York and you're meeting people from around the world like the two of you. That's number one. Uh number two is uh it's part of the harbor. It's part of the ecosystem here. It's the estuary. You know, on good days, you walk out or bike out to the Hudson River and you smell the ocean. And the third is uh you know, it's a cultural capital of the western world. So, [Music] The Big Apple's most famous street is the Fifth Avenue. It crosses Manhattan from north to south, and it's one of the most expensive streets in the world. So many interesting things here. World's famous Apple store, Trump Tower with BLM mural in front of it, flat iron building, St. Patrick's Cathedral, a gorgeous Catholic cathedral that was built in 1878, Empire State Building, and so much more. A section of Fifth Avenue running from 34th to 

# **Step 3 - Augmentation**

In [62]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-4-31B-it",
    task='text-generation',
    max_new_tokens=512
)

model = ChatHuggingFace(llm=llm)

In [69]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer only from the provided transcript context.
    If the contect is insufficient, just say you don't know.

    {context}
    Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [74]:
question = "is the topic of woman discussed in this video?"
retrieved_docs = retriever.invoke(question)

In [75]:
context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
context_text

"of the park, resembling the interior of the old refinery. New York City is a city of hundreds of different languages and a variety of communities. Yes, it's a melting pot, but only to a certain extent. Most of its neighborhoods are dominated by a certain ethnicity. You know, it makes sense because people love to be around other people that share a similar cultural background. Let's visit a Jewish area in Brooklyn called Burrow Park. First thing you notice is the abundance of young mothers with strollers. The fertility rate for Orthodox Jews is one of the highest in the world. An average of 6.5 children per woman. For comparison, in Europe is 1.6. Let's talk to some locals. Uh what part of town are we in? You're in the Burough Park section of Brooklyn. And what's it famous for? Jewish neighborhood. What is it that you like mostly about living here? Well, it's the community. you know, it's we're tightknit, we're friendly to each other, and we know each other. And it's also this pretty\n

In [76]:
final_prompt = prompt.invoke({"context":context_text, "question": question})

# **Step - 4 Generation**

In [77]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the topic of women is discussed in the following ways:
* In Borough Park, the fertility rate for Orthodox Jews is mentioned as one of the highest in the world, with an average of 6.5 children per woman.
* It is mentioned that married women cover their hair.
* Carrie Bradshaw from "Sex in the City" is mentioned as having lived in the Village.


# Building a Chain

In [80]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [84]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [85]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [86]:
parallel_chain.invoke("what is manhattan")

{'context': "Manhattan is full of contrast. $2 hot dog stands stay side by side with expensive boutiques on Fifth Avenue. Extremely wealthy people are walking the same streets as the homeless. Some people might get the impression that Manhattan only has Central Park and the rest is nothing but a concrete jungle, but it's not like that. There are green areas scattered all throughout the island like Brian Park located next to the New York Public Library main branch. It has a huge lawn where people are reading newspapers, playing pingpong, chess, and bag gaming. In the winter, they will have a skating rink here. [Music] Another great park is Madison Square Park. It's located at the intersection of Broadway and the Fifth Avenue overlooking the Flat Iron Building. It's a great place to unwind and enjoy a gelato on a hot summer day or a cup of coffee in the fall. One of the most exciting additions to the city is the High Line. It was created in 2009 and it's 1.4 mile long lawn elevated green

In [87]:
parser = StrOutputParser()

In [88]:
main_chain = parallel_chain | prompt | model | parser

In [89]:
main_chain.invoke('can you summarize the video')

"The video explores various locations in New York City. The narrator visits Central Park, describing it as a man-made, highly visited urban park and popular film location that features a zoo, artificial ponds, ice rinks, and the 3,500-year-old obelisk known as Cleopatra's Needle. \n\nThe narrator also visits Harlem, noting its history and gentrification, and explores Chinatown and the nearby Little Italy, which is described as having become very touristy. Additionally, the video mentions Chelsea as an art district with a large LGBTQ population. The video concludes with facts about New York City, noting that it has more billionaires than any other city in the world and the largest foreign-born population of any city."